In [11]:
import muon as mu
from muon import atac as ac
import scanpy as sc
import seaborn as sns
from scipy.stats import median_abs_deviation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
atac = sc.read_h5ad('../data/atac_subset.h5ad')

In [9]:
atac

AnnData object with n_obs × n_vars = 3129 × 218882
    obs: 'sample_id', 'Neurotypical reference', 'Donor ID', 'Organism', 'Brain Region', 'Sex', 'Gender', 'Age at Death', 'Race (choice=White)', 'Race (choice=Black/ African American)', 'Race (choice=Asian)', 'Race (choice=American Indian/ Alaska Native)', 'Race (choice=Native Hawaiian or Pacific Islander)', 'Race (choice=Unknown or unreported)', 'Race (choice=Other)', 'specify other race', 'Hispanic/Latino', 'Highest level of education', 'Years of education', 'PMI', 'Fresh Brain Weight', 'Brain pH', 'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score', 'Overall CAA Score', 'Highest Lewy Body Disease', 'Total Microinfarcts (not observed grossly)', 'Total microinfarcts in screening sections', 'Atherosclerosis', 'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Last CASI Score', 'Interval from last CASI in months', 'Last MMSE Score', 'Interval from last MMSE in months', 'Last MOCA Score', 'Interval from last MOCA in mont

# QC (already done for sea-ad)

In [10]:
# Calculate general qc metrics using scanpy
sc.pp.calculate_qc_metrics(atac, percent_top=None, log1p=False, inplace=True)

# Rename columns
atac.obs.rename(
    columns={
        "n_genes_by_counts": "n_features_per_cell",
        "total_counts": "total_fragment_counts",
    },
    inplace=True,
)

# log-transform total counts and add as column
atac.obs["log_total_fragment_counts"] = np.log10(atac.obs["total_fragment_counts"])

In [13]:
atac

AnnData object with n_obs × n_vars = 3129 × 218882
    obs: 'sample_id', 'Neurotypical reference', 'Donor ID', 'Organism', 'Brain Region', 'Sex', 'Gender', 'Age at Death', 'Race (choice=White)', 'Race (choice=Black/ African American)', 'Race (choice=Asian)', 'Race (choice=American Indian/ Alaska Native)', 'Race (choice=Native Hawaiian or Pacific Islander)', 'Race (choice=Unknown or unreported)', 'Race (choice=Other)', 'specify other race', 'Hispanic/Latino', 'Highest level of education', 'Years of education', 'PMI', 'Fresh Brain Weight', 'Brain pH', 'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score', 'Overall CAA Score', 'Highest Lewy Body Disease', 'Total Microinfarcts (not observed grossly)', 'Total microinfarcts in screening sections', 'Atherosclerosis', 'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Last CASI Score', 'Interval from last CASI in months', 'Last MMSE Score', 'Interval from last MMSE in months', 'Last MOCA Score', 'Interval from last MOCA in mont

In [14]:
atac.obs

,sample_id,Neurotypical reference,Donor ID,Organism,Brain Region,Sex,Gender,Age at Death,Race (choice=White),Race (choice=Black/ African American),...,Supertype confidence,Supertype (non-expanded),Supertype,RNA Quality Control Score,Quality Control Clusters,Continuous Pseudo-progression Score,Severely Affected Donor,n_features_per_cell,total_fragment_counts,log_total_fragment_counts
index,,,,,,,,,,,,,,,,,,,,,
GGTGAGGTCAATAGCC-L8XR_210729_01_E09-1122543707,GGTGAGGTCAATAGCC-L8XR_210729_01_E09-1122543707,False,H20.33.020,human,Human MTG,Male,Male,81.0,Checked,Unchecked,...,0.999998,Micro-PVM_2,Micro-PVM_3-SEAAD,0.105263,12,0.928513,N,4096,9503.0,3.977861
CTAGGACGTTAGTTGG-L8XR_211007_02_F03-1135448413,CTAGGACGTTAGTTGG-L8XR_211007_02_F03-1135448413,False,H21.33.001,human,Human MTG,Male,Male,80.0,Checked,Unchecked,...,0.999558,Micro-PVM_2,Micro-PVM_2,0.125000,12,0.149616,N,2520,5657.0,3.752586
GGTCCTGCACACTAAT-L8XR_210812_01_E11-1126220036,GGTCCTGCACACTAAT-L8XR_210812_01_E11-1126220036,False,H20.33.039,human,Human MTG,Female,Female,96.0,Checked,Unchecked,...,1.000000,Micro-PVM_2,Micro-PVM_2,0.062500,12,0.371467,N,11336,35703.0,4.552705
AATTAGCGTTGGCCGA-L8XR_211021_02_G06-1138433804,AATTAGCGTTGGCCGA-L8XR_211021_02_G06-1138433804,False,H20.33.043,human,Human MTG,Male,Male,85.0,Checked,Unchecked,...,0.999995,Micro-PVM_2,Micro-PVM_2,0.052632,12,0.442128,N,9461,27889.0,4.445433
AGTCAGGCATGAATAG-L8XR_210715_01_A12-1121939868,AGTCAGGCATGAATAG-L8XR_210715_01_A12-1121939868,False,H20.33.014,human,Human MTG,Female,Female,82.0,Checked,Unchecked,...,1.000000,Micro-PVM_2,Micro-PVM_2,0.137931,12,0.678277,N,8035,22896.0,4.359760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATCAAGCTCCTTCGTA-L8XR_210812_01_A12-1124987483,ATCAAGCTCCTTCGTA-L8XR_210812_01_A12-1124987483,False,H20.33.001,human,Human MTG,Male,Male,82.0,Checked,Unchecked,...,1.000000,Micro-PVM_2,Micro-PVM_2_3-SEAAD,0.157895,12,0.522889,N,470,919.0,2.963315
ACAACAGAGGGTCTAT-L8XR_211014_02_H05-1136687627,ACAACAGAGGGTCTAT-L8XR_211014_02_H05-1136687627,False,H21.33.008,human,Human MTG,Female,Female,91.0,Checked,Unchecked,...,1.000000,Micro-PVM_2,Micro-PVM_2_3-SEAAD,0.000000,12,0.790347,N,9920,27358.0,4.437084
TTGGGCCAGACTTATG-L8XR_210812_01_A12-1124987483,TTGGGCCAGACTTATG-L8XR_210812_01_A12-1124987483,False,H20.33.001,human,Human MTG,Male,Male,82.0,Checked,Unchecked,...,1.000000,Micro-PVM_2,Micro-PVM_2,0.043478,12,0.522889,N,1190,2451.0,3.389343
